# 1. Family example

This walkthrough follows exactly one fixed raw record:

- file: `data/raw/family/FamilyOWL_1hop.json`
- group index: `0`
- QA index: `0`
- task: `1hop-Thing_alan_john_dowse_1936_...-rdf:type-Man-BIN`
- question: **Is Alan John Dowse a man?**

No other FamilyOWL record is selected or processed.

In [16]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data.build_subgraph_training_data import (
    beam_connected_subgraphs,
    get_gold_explanations,
    materialize_retrieval_rows,
    parse_owl_context,
    relevant_context_axioms,
    set_scores,
)

## What the raw FamilyOWL dataset contains

The JSON root is a list of ontology-centered groups. Each group contains one ontology and a `QAs` list.

| Raw field | SAGE-QA use for this example |
|---|---|
| group `OWL Context` | **Inference input:** parse into candidate facts and axioms |
| QA `NL Question` | **Inference input:** natural-language query for the retriever and reader |
| QA `SPARQL Query` | **Inference input:** identify query entities/properties for ontology alignment |
| QA `Explanations` | **Gold supervision/evaluation only:** target reasoning paths |
| QA `Answer` | **Gold answer supervision/evaluation only** |
| group `Task Type`, `Answer Type`, `Root Entity`; QA `Task ID` | Metadata and diagnostics |
| group `NL Context`, `ABS Context`; QA `ABS Question`, `ABS Answer` | Not used in this concrete SAGE-QA path |

In [17]:
raw_path = REPO_ROOT / "data/raw/family/FamilyOWL_1hop.json"
groups = json.loads(raw_path.read_text(encoding="utf-8"))
development_metadata_path = (
    REPO_ROOT / "data/development/familyowl_v3/FamilyOWL_1hop/metadata.json"
)
full_metadata_path = REPO_ROOT / "data/FamilyOWL_1hop/metadata.json"
retrieval_metadata_path = (
    development_metadata_path
    if development_metadata_path.exists()
    else full_metadata_path
)
retrieval_metadata = (
    json.loads(retrieval_metadata_path.read_text(encoding="utf-8"))
    if retrieval_metadata_path.exists()
    else {}
)

GROUP_INDEX = 0
QA_INDEX = 0
REFERENCE_TASK_ID = "1hop-Thing_alan_john_dowse_1936_alan_john_dowse_1936-alan_john_dowse_1936-rdf:type-Man-BIN"

group = groups[GROUP_INDEX]
qa = group["QAs"][QA_INDEX]
assert qa["Task ID"] == REFERENCE_TASK_ID

print("Raw file:", raw_path.relative_to(REPO_ROOT))
print("Reference:", {"group_index": GROUP_INDEX, "qa_index": QA_INDEX})
print("Raw group fields:", list(group))
print("Raw QA fields:", list(qa))
print("Task ID:", qa["Task ID"])

Raw file: data\raw\family\FamilyOWL_1hop.json
Reference: {'group_index': 0, 'qa_index': 0}
Raw group fields: ['Task Type', 'Answer Type', 'Root Entity', 'OWL Context', 'NL Context', 'ABS Context', 'QAs']
Raw QA fields: ['Task ID', 'SPARQL Query', 'NL Question', 'ABS Question', 'ABS Answer', 'Answer', 'Minimum Explanation', 'Explanations', 'Explanation Count', 'Explanation Min', 'Explanation Max']
Task ID: 1hop-Thing_alan_john_dowse_1936_alan_john_dowse_1936-alan_john_dowse_1936-rdf:type-Man-BIN


## Separate inference inputs from gold labels

In [18]:
# Available to SAGE-QA when ranking/answering this example.
question = qa["NL Question"]
reference_sparql_query = qa["SPARQL Query"]
inference_sparql_query = ""  # Gold SPARQL is never exposed to retrieval.
owl_context = group["OWL Context"]

# Used only to train or evaluate the prediction.
gold_answer = qa["Answer"]
gold_explanations = get_gold_explanations(qa)

print("Inference question:", question)
print("Reference SPARQL (evaluation only):", reference_sparql_query)
print("Gold answer:", gold_answer)
print("Gold reasoning path:")
print(*gold_explanations[0], sep="\n")

Inference question: Is Alan John Dowse a man?
Reference SPARQL (evaluation only): ASK WHERE { <http://www.example.com/genealogy.owl#alan_john_dowse_1936> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.example.com/genealogy.owl#Man> }
Gold answer: TRUE
Gold reasoning path:
martin_dowse_1944 hasBrother alan_john_dowse_1936
InverseObjectProperties(hasBrother,isBrotherOf)
isBrotherOf domain Man


## Build the candidate KG from `OWL Context`

FamilyOWL is ontology-native, so no text-to-KG model is needed:

```text
OWL Context
    │ parse
    ▼
independent candidate triples/axioms
    │ beam composition
    ▼
candidate reasoning paths
    │ GNN ranking
    ▼
predicted reasoning path

Gold Explanations ──► training supervision and evaluation only
```

Neither parsing nor graph-neighborhood selection receives the gold explanation.

In [19]:
# Candidate triples/axioms from the OWL context that are relevant to the question/SPARQL query.

all_context_units = parse_owl_context(owl_context)
candidate_units = relevant_context_axioms(
    all_context_units,
    question=question,
    sparql_query=inference_sparql_query,
    max_context_units=40,
)

candidate_set = set(candidate_units)
gold_context_coverage = max(
    len(set(gold) & candidate_set) / len(set(gold)) for gold in gold_explanations
)

print("Parsed OWL facts/axioms:", len(all_context_units))
print("Selected candidate KG units:", len(candidate_units))
print("Gold context coverage (diagnostic only):", gold_context_coverage)
print("First candidate units:")
print(*candidate_units[:12], sep="\n")

Parsed OWL facts/axioms: 92
Selected candidate KG units: 40
Gold context coverage (diagnostic only): 1.0
First candidate units:
alan_john_dowse_1936 hasMother ethel_archer_1912
martin_dowse_1944 hasBrother alan_john_dowse_1936
maureen_dowse_1939 hasBrother alan_john_dowse_1936
Man disjointWith Marriage
Man disjointWith Woman
hasFather range Man
hasHusband range Man
hasMalePartner range Man
isBrotherOf domain Man
isUncleOf domain Man
FunctionalObjectProperty(hasFather)
FunctionalObjectProperty(hasMother)


## Compose candidate reasoning paths

The `candidate_units` above are independent triples or axioms. `beam_connected_subgraphs` combines them into bounded, connected candidate reasoning paths for the GNN to rank.

During this search, an internal heuristic score keeps combinations that are structurally connected and relevant to the question. Its only purpose is to control the beam and avoid enumerating every possible combination. It is not the GNN score and does not use the gold explanation.

After all candidates have been generated, `set_scores` compares them with the gold explanation for supervision and evaluation. This post-hoc comparison does not influence candidate construction.

In [20]:
# Compose the independent candidate units into bounded, connected combinations.
# Gold explanations are not provided to this function.

candidate_subgraphs = beam_connected_subgraphs(
    candidate_units=candidate_units,
    question=question,
    sparql_query=inference_sparql_query,
    min_subgraph_size=1,
    max_subgraph_size=(
        min(7, len(candidate_units))
        if int(retrieval_metadata.get("max_subgraph_size", 0)) <= 0
        else int(retrieval_metadata["max_subgraph_size"])
    ),
    beam_width=int(retrieval_metadata.get("candidate_beam_width", 96)),
    max_candidate_subgraphs=int(retrieval_metadata.get("max_candidate_subgraphs", 320)),
)

# Post-hoc oracle scoring for supervision/evaluation only.
oracle_scored_candidates = sorted(
    (
        (set_scores(list(candidate), gold_explanations), candidate)
        for candidate in candidate_subgraphs
    ),
    key=lambda item: item[0]["best_set_f1_to_gold"],
    reverse=True,
)
oracle_best_scores, oracle_best_candidate = oracle_scored_candidates[0]
print("Generated candidate paths:", len(candidate_subgraphs))
print("Oracle-best candidate for this reference example:")
print(*oracle_best_candidate, sep="\n")
print("Supervision/evaluation metrics:", oracle_best_scores)

Generated candidate paths: 309
Oracle-best candidate for this reference example:
martin_dowse_1944 hasBrother alan_john_dowse_1936
isBrotherOf domain Man
InverseObjectProperties(hasBrother,isBrotherOf)
Supervision/evaluation metrics: {'best_jaccard_to_gold': 1.0, 'best_set_precision_to_gold': 1.0, 'best_set_recall_to_gold': 1.0, 'best_set_f1_to_gold': 1.0, 'exact_match_any_gold': True, 'contains_any_gold_explanation': True, 'contained_in_any_gold_explanation': True, 'best_matching_gold_explanation': ['martin_dowse_1944 hasBrother alan_john_dowse_1936', 'InverseObjectProperties(hasBrother,isBrotherOf)', 'isBrotherOf domain Man'], 'best_matching_gold_index': 0}


## Materialize the GNN retrieval rows

The GNN scores candidate subgraphs, not the independent `candidate_units` directly. The shared `materialize_retrieval_rows` function therefore creates one row for each generated candidate path. The 2Wiki walkthrough uses this same function after its text-to-KG step.

- `subgraph_units` and `subgraph_node_ids` identify the candidate presented to the model.
- `label` says whether the candidate contains a complete gold explanation.
- `best_set_f1_to_gold` measures the candidate's triple-set overlap with the closest valid gold explanation.

For a candidate set $C$ and gold explanation set $G$, precision is $|C \cap G|/|C|$, recall is $|C \cap G|/|G|$, and Gold F1 is their harmonic mean. If several gold explanations are valid, the maximum F1 is retained—hence **best** set F1. During training, it contributes to the GNN ranking target; it is not an input feature. These gold-derived fields provide training supervision and evaluation diagnostics and are unavailable during real inference.

In [21]:
family_example_id = (
    f"FamilyOWL_1hop__g{GROUP_INDEX}__q{QA_INDEX}__{qa.get('Task ID', '')}__{question}"
)
retrieval_rows = materialize_retrieval_rows(
    candidate_units=candidate_units,
    candidate_subgraphs=candidate_subgraphs,
    gold_explanations=gold_explanations,
    base_row={
        "example_id": family_example_id,
        "dataset": "FamilyOWL_1hop",
        "question": question,
        "sparql_query": inference_sparql_query,
        "reference_sparql_query": reference_sparql_query,
        "answer": qa["Answer"],
        "answer_type": group["Answer Type"],
        "evidence_unit_type": "ontology_axiom",
        "gold_explanations": gold_explanations,
        "gold_units": gold_explanations[0],
        "gold_context_coverage": gold_context_coverage,
    },
    max_negative_per_example=int(
        retrieval_metadata.get("max_negative_per_example", 200)
    ),
    shuffle_rows=False,
)

assert len({row["example_id"] for row in retrieval_rows}) == 1
assert any(row["exact_match_any_gold"] for row in retrieval_rows), (
    "The exact walkthrough proof is missing from the candidate pool. "
    "Restart the kernel and run all cells so the current candidate builder is loaded."
)
print("Rows for this one question:", len(retrieval_rows))
print("Positive training rows:", sum(row["label"] for row in retrieval_rows))
print("Example candidate row:")
print(
    json.dumps(
        {
            key: retrieval_rows[0][key]
            for key in (
                "subgraph_node_ids",
                "subgraph_units",
                "symbolic_features",
                "label",
                "best_set_f1_to_gold",
            )
            if key in retrieval_rows[0]
        },
        indent=2,
    )
)

Rows for this one question: 247
Positive training rows: 47
Example candidate row:
{
  "subgraph_node_ids": [
    1,
    8,
    12
  ],
  "subgraph_units": [
    "martin_dowse_1944 hasBrother alan_john_dowse_1936",
    "isBrotherOf domain Man",
    "InverseObjectProperties(hasBrother,isBrotherOf)"
  ],
  "label": 1,
  "best_set_f1_to_gold": 1.0
}


## Prepare the shared local graph

`prepare_examples` reconstructs the local node universe as the union of all candidate rows and maps every candidate path to node IDs. For this one question, the GNN will encode the question and nodes once, run message passing over the shared graph, then pool a different node subset for each candidate.

In [22]:
import torch

from models.gnn_subgraph_retriever import GNNSubgraphRetriever
from training.train_gnn_subgraph_retriever import (
    encode_example_graph,
    prepare_examples,
    score_candidate_rows,
)
from utils.tokenizer import load_tokenizer

prepared_examples = prepare_examples(retrieval_rows, subsample_candidates=False)
assert len(prepared_examples) == 1
gnn_example = prepared_examples[0]

print("Local GNN nodes:", len(gnn_example["candidate_axioms"]))
print("Candidate subgraphs to score:", len(gnn_example["candidate_rows"]))
print(
    "First candidate node IDs:", gnn_example["candidate_rows"][0]["subgraph_node_ids"]
)

Local GNN nodes: 33
Candidate subgraphs to score: 247
First candidate node IDs: [0, 1, 2]


## Run GNN retrieval

This cell loads the local FamilyOWL checkpoint, creates the full message-passing graph, and assigns one learned GNN probability to each candidate subgraph. At inference time, candidates are ranked only by this probability.

The printed **Gold F1** is the `best_set_f1_to_gold` metric defined above. It is shown beside each ranked prediction only so that we can inspect retrieval quality. It is not a model prediction and is not passed to the model during inference.

The existing checkpoint may predate the current inference-safe candidate builder. Rebuild FamilyOWL and retrain before using its scores as manuscript results; here it makes the retrieval mechanics executable and inspectable.

In [23]:
development_checkpoint = (
    REPO_ROOT / "checkpoints/development/gnn_familyowl_1hop_nl_only_exact/best_model.pt"
)
full_checkpoint = (
    REPO_ROOT / "checkpoints/gnn_subgraph_ranker_familyowl_1hop_full/best_model.pt"
)
CHECKPOINT = (
    development_checkpoint if development_checkpoint.exists() else full_checkpoint
)
RUN_GNN = CHECKPOINT.exists()

if not RUN_GNN:
    print("Checkpoint not found; expected:", CHECKPOINT)
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
    model_name = checkpoint["model_name"]
    tokenizer = load_tokenizer(model_name)
    model = GNNSubgraphRetriever(
        model_name=model_name,
        node_symbolic_dim=checkpoint.get("node_symbolic_dim", 8),
        subgraph_symbolic_dim=checkpoint.get("subgraph_symbolic_dim", 8),
        gnn_hidden_dim=checkpoint.get("gnn_hidden_dim", 128),
        gnn_layers=checkpoint.get("gnn_layers", 2),
        classifier_hidden_dim=checkpoint.get("classifier_hidden_dim", 128),
        freeze_encoder=checkpoint.get("freeze_encoder", False),
        architecture_version=checkpoint.get("architecture_version", 1),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    with torch.no_grad():
        encoded_graph = encode_example_graph(
            model=model,
            tokenizer=tokenizer,
            example=gnn_example,
            device=device,
            max_length=128,
        )
        ranked_rows = []
        for start in range(0, len(gnn_example["candidate_rows"]), 64):
            batch = gnn_example["candidate_rows"][start : start + 64]
            output = score_candidate_rows(
                model=model,
                encoded_graph=encoded_graph,
                candidate_rows=batch,
                device=device,
            )
            for row, probability in zip(batch, output["probs"].cpu().tolist()):
                ranked_rows.append({**row, "gnn_score": float(probability)})

    ranked_rows.sort(key=lambda row: row["gnn_score"], reverse=True)
    print("Checkpoint:", CHECKPOINT.relative_to(REPO_ROOT))
    print("Training objective:", checkpoint.get("training_objective", "legacy"))
    print("Device:", device)
    print("Encoder:", model_name)
    print("Local graph nodes:", len(gnn_example["candidate_axioms"]))
    print("Directed edges including self-loops:", encoded_graph["edge_index"].shape[1])
    print("Top-5 GNN retrieval results:")
    for rank, row in enumerate(ranked_rows[:5], start=1):
        print(
            f"\nRank {rank} | score={row['gnn_score']:.6f} | units={len(row['subgraph_units'])}"
        )
        print(*row["subgraph_units"], sep="\n")
        print("Gold F1 (evaluation only):", row["best_set_f1_to_gold"])

    exact_ranked = [
        (rank, row)
        for rank, row in enumerate(ranked_rows, start=1)
        if row["exact_match_any_gold"]
    ]
    oracle_f1 = max(row["best_set_f1_to_gold"] for row in ranked_rows)
    print("\nWalkthrough reference diagnostics:")
    print("Candidate-pool oracle F1:", oracle_f1)
    if exact_ranked:
        exact_rank, exact_row = exact_ranked[0]
        print("Best exact-proof rank:", exact_rank)
        print("Best exact-proof score:", round(exact_row["gnn_score"], 6))
        print(
            "Rank-1 minus exact score gap:",
            round(ranked_rows[0]["gnn_score"] - exact_row["gnn_score"], 6),
        )
    else:
        print("Best exact-proof rank: unavailable (candidate-generation failure)")

Checkpoint: checkpoints\development\gnn_familyowl_1hop_nl_only_exact\best_model.pt
Training objective: within_question_pairwise_plus_exact_contrastive
Device: cpu
Encoder: google/bert_uncased_L-2_H-128_A-2
Local graph nodes: 33
Directed edges including self-loops: 249
Top-5 GNN retrieval results:

Rank 1 | score=0.510577 | units=7
alan_john_dowse_1936 hasMother ethel_archer_1912
martin_dowse_1944 hasBrother alan_john_dowse_1936
maureen_dowse_1939 hasBrother alan_john_dowse_1936
hasFather range Man
InverseObjectProperties(hasFather,isFatherOf)
SubObjectPropertyOf(hasFather,hasParent)
SubObjectPropertyOf(hasMother,hasParent)
Gold F1 (evaluation only): 0.2

Rank 2 | score=0.510568 | units=5
alan_john_dowse_1936 hasMother ethel_archer_1912
martin_dowse_1944 hasBrother alan_john_dowse_1936
InverseObjectProperties(hasFather,isFatherOf)
SubObjectPropertyOf(hasFather,hasParent)
SubObjectPropertyOf(hasMother,hasParent)
Gold F1 (evaluation only): 0.25

Rank 3 | score=0.510547 | units=7
alan_john

In [24]:
# Top 3 ranks are used as input for the downstream answer generation model.